# Notebook 2 — Evaluation
**MedThink-Bench VI Eval Pipeline**

Input : `checkpoint_inference.jsonl` (từ Notebook 1A hoặc 1B hoặc 1C)  
Output: `results_eval.jsonl` + `summary_metrics.json`

**3 Metrics:**
- **Accuracy** — hard, objective: model chọn đúng MCQ label không?
- **Rationale Recall** — soft, semi-objective: reasoning có cover đủ key medical facts không?
- **BertScore** — soft, objective: semantic similarity của reasoning vs scoring_points (PhoBERT tiếng Việt)

---
**Yêu cầu**: `pip install bert-score aiohttp`

In [1]:
# ── Cell 1: Install dependencies ────────────────────────────────────────
!pip install bert-score aiohttp -q
!pip install bert-score sentencepiece -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.0 MB/s eta 0:00:00


In [2]:
# ── Cell 1.5: Copy shared modules từ dataset vào working dir ────────────
import shutil, os

_SRC = "/kaggle/input/datasets/quangminh2401/medthink-vi-eval-pipeline"
_DST = "/kaggle/working"

for _f in ["utils.py", "async_openrouter.py", "evaluator.py","report_generator.py"]:
    shutil.copy(os.path.join(_SRC, _f), os.path.join(_DST, _f))
    print(f"copied: {_f}")

copied: utils.py
copied: async_openrouter.py
copied: evaluator.py
copied: report_generator.py


In [3]:
# ── Cell 2: CONFIG - Điền thông tin cấu hình tại đây ───────────────────────────────────────────────────────
import sys
sys.path.insert(0, "/kaggle/working")

# Load API key từ Kaggle Secrets (Settings → Secrets → OPENROUTER_API_KEY)
from kaggle_secrets import UserSecretsClient
OPENROUTER_API_KEY = UserSecretsClient().get_secret("OPENROUTER_API_KEY")


# ── Đường dẫn file ──────────────────────────
CHECKPOINT_INFER    = "/kaggle/input/datasets/quangminh2401/medthink-bnechclaude-sonnet-4-6/inference_results.jsonl"    # Resume inference
CHECKPOINT_EVAL     = "/kaggle/working/checkpoint_eval.jsonl"         # Resume eval
OUTPUT_EVAL         = "/kaggle/working/results_eval.jsonl"            # Kết quả per-sample
OUTPUT_SUMMARY      = "/kaggle/working/summary_metrics.json"          # Aggregate report

# ── OpenRouter API ───────────────────────────
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1/chat/completions"

# ── LLM Judge (Notebook 2 – OpenRouter) ─────
JUDGE_MODEL         = "google/gemini-2.5-flash"                   # Model judge
JUDGE_MAX_TOKENS    = 4096
JUDGE_TEMPERATURE   = 0.0
JUDGE_SEMAPHORE     = 5         
JUDGE_RETRY_MAX     = 3
JUDGE_TIMEOUT_SEC   = 90

# ── BertScore ────────────────────────────────
BERTSCORE_MODEL     = "vinai/phobert-base-v2"                         # PhoBERT tiếng Việt
BERTSCORE_FALLBACK  = "bert-base-multilingual-cased"                  # Fallback nếu lỗi
BERTSCORE_BATCH_SIZE = 32
BERTSCORE_LANG      = "vi"

# ── Prompt LLM Judge (batch scoring_points) ──
JUDGE_SYSTEM_PROMPT = """Bạn là một chuyên gia đánh giá câu trả lời y khoa.

Cho một câu hỏi, một danh sách các điểm chấm điểm (scoring points) cần kiểm tra,
và phần lý luận của model, hãy đánh giá xem model có đề cập đến từng điểm hay không.

Quy tắc đánh giá (point-wise):
- Một điểm được tính là "đề cập" CHỈ KHI nội dung chính của nó được phát biểu RÕ RÀNG trong phần lý luận.
- KHÔNG tính nếu thiếu thông tin quan trọng của điểm đó.
- KHÔNG tính nếu thông tin trong lý luận MÂU THUẪN với điểm đó.
- KHÔNG tính nếu chỉ đề cập một phần nhỏ của điểm đó.

Trả lời ĐÚNG theo định dạng JSON sau, không thêm bất kỳ nội dung nào khác:
{
  "results": [
    {"point_index": 0, "contains": true/false, "reason": "<giải thích ngắn gọn>"},
    {"point_index": 1, "contains": true/false, "reason": "<giải thích ngắn gọn>"},
    ...
  ]
}"""

JUDGE_USER_TEMPLATE = """## Câu hỏi:
{question}

## Các điểm cần đánh giá (scoring points):
{scoring_points_numbered}

## Lý luận của model:
{reasoning}"""


print(f"Input          : {CHECKPOINT_INFER}")
print(f"Judge model    : {JUDGE_MODEL}")
print(f"BertScore model: {BERTSCORE_MODEL}")
print(f"Output eval    : {OUTPUT_EVAL}")
print(f"Output summary : {OUTPUT_SUMMARY}")

Input          : /kaggle/input/datasets/quangminh2401/medthink-bnechclaude-sonnet-4-6/inference_results.jsonl
Judge model    : google/gemini-2.5-flash
BertScore model: vinai/phobert-base-v2
Output eval    : /kaggle/working/results_eval.jsonl
Output summary : /kaggle/working/summary_metrics.json


In [4]:
# ── Cell 3: Load inference results + resume checkpoint ──────────────────
from utils import load_jsonl, load_checkpoint_indices

all_records  = load_jsonl(CHECKPOINT_INFER)

# Gán lại _original_position theo thứ tự trong file
# (file đã được sort ở bước xuất inference)
for i, r in enumerate(all_records):
    r["_original_position"] = i

done_indices = load_checkpoint_indices(CHECKPOINT_EVAL)
todo_records = [r for r in all_records if r["index"] not in done_indices]

n_null = sum(1 for r in todo_records if r.get("raw_output") is None)
print(f"Tổng inference records : {len(all_records)}")
print(f"Đã eval (checkpoint)   : {len(done_indices)}")
print(f"Còn lại                : {len(todo_records)}")
print(f"Null output (skip judge): {n_null}")

Tổng inference records : 500
Đã eval (checkpoint)   : 0
Còn lại                : 500
Null output (skip judge): 0


In [5]:
# ── Cell 4: Accuracy (sync, instant) ───────────────────────────────────
# Tính trên toàn bộ all_records (bao gồm cả records đã eval từ run trước)
from evaluator import compute_accuracy

acc = compute_accuracy(all_records)

print("── Accuracy ──────────────────────────────")
print(f"  Accuracy : {acc['accuracy']:.4f}")
print(f"  Correct  : {acc['n_correct']} / {acc['n_total']}")
print(f"  Skipped  : {acc['n_skipped']} (null hoặc không parse được prediction)")

── Accuracy ──────────────────────────────
  Accuracy : 0.5912
  Correct  : 295 / 499
  Skipped  : 1 (null hoặc không parse được prediction)


In [6]:
# ── Cell 5: BertScore (batch, PhoBERT) ─────────────────────────────────
# PhoBERT ~400MB — lần đầu sẽ download, sau đó cache tại ~/.cache/huggingface
# Fallback tự động sang bert-base-multilingual-cased nếu PhoBERT lỗi
from evaluator import compute_bertscore_batch

print(f"Đang tính BertScore với {BERTSCORE_MODEL}...")
print(f"Fallback: {BERTSCORE_FALLBACK}")

bertscore_scores = compute_bertscore_batch(
    records        = todo_records,
    model_type     = BERTSCORE_MODEL,
    lang           = BERTSCORE_LANG,
    batch_size     = BERTSCORE_BATCH_SIZE,
    fallback_model = BERTSCORE_FALLBACK,
)

n_valid = sum(1 for s in bertscore_scores if s is not None)
mean_bs = sum(s for s in bertscore_scores if s is not None) / n_valid if n_valid else 0.0
print(f"\nBertScore F1 (mean): {mean_bs:.4f}  ({n_valid}/{len(todo_records)} samples có score)")

Đang tính BertScore với vinai/phobert-base-v2...
Fallback: bert-base-multilingual-cased


04:46:42 [INFO] NumExpr defaulting to 4 threads.
04:47:01 [INFO] BertScore: dùng model vinai/phobert-base-v2
04:47:01 [WARNING] BertScore với vinai/phobert-base-v2 thất bại: 'vinai/phobert-base-v2'
04:47:01 [INFO] BertScore: dùng model bert-base-multilingual-cased
04:47:02 [INFO] HTTP Request: HEAD https://huggingface.co/bert-base-multilingual-cased/resolve/main/config.json "HTTP/1.1 200 OK"
04:47:02 [INFO] HTTP Request: GET https://huggingface.co/bert-base-multilingual-cased/resolve/main/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

04:47:02 [INFO] HTTP Request: HEAD https://huggingface.co/bert-base-multilingual-cased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
04:47:02 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
04:47:02 [INFO] HTTP Request: GET https://huggingface.co/bert-base-multilingual-cased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/32 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/16 [00:00<?, ?it/s]

done in 397.65 seconds, 1.26 sentences/sec

BertScore F1 (mean): 0.7194  (500/500 samples có score)


In [7]:
# ── Cell 6: LLM Judge — Rationale Recall (async) ────────────────────────
# Option A: 1 API call per sample, judge làm membership check
# Score = n_recalled / n_scoring_points  →  Rationale Recall
import asyncio
from tqdm.notebook import tqdm
from async_openrouter import AsyncOpenRouterClient
from evaluator import run_judge_async
from utils import TokenTracker

judge_tracker = TokenTracker(label="LLM Judge")

async def run_judge_all():
    async with AsyncOpenRouterClient(
        api_key         = OPENROUTER_API_KEY,
        base_url        = OPENROUTER_BASE_URL,
        model           = JUDGE_MODEL,
        max_tokens      = JUDGE_MAX_TOKENS,
        temperature     = JUDGE_TEMPERATURE,
        semaphore_limit = JUDGE_SEMAPHORE,
        retry_max       = JUDGE_RETRY_MAX,
        timeout_sec     = JUDGE_TIMEOUT_SEC,
        tracker         = judge_tracker,
    ) as client:
        with tqdm(total=len(todo_records), desc="LLM Judge") as pbar:
            results = await run_judge_async(
                records       = todo_records,
                client        = client,
                system_prompt = JUDGE_SYSTEM_PROMPT,
                user_template = JUDGE_USER_TEMPLATE,
                pbar          = pbar,
            )
    return results

judge_results = await run_judge_all()

# ── In tổng token Judge sau khi cell chạy xong ──
judge_tracker.print_summary()

LLM Judge:   0%|          | 0/500 [00:00<?, ?it/s]


  [LLM Judge] Token Usage Summary
  API calls       : 500
  Prompt tokens   : 527,026
  Completion tokens: 121,080
  Total tokens    : 648,106
  Elapsed time    : 155.9s
  Avg tokens/call : 1,296



In [11]:
# ── Cell 7: Ghi checkpoint eval ─────────────────────────────────────────
from utils import append_jsonl, extract_ground_truth_label

n_parse_errors = 0

for record, bs_score, judge_result in zip(todo_records, bertscore_scores, judge_results):
    gt   = extract_ground_truth_label(record.get("answer", ""))
    pred = (record.get("extracted_answer") or "").upper()

    # **record đã chứa _original_position được gán ở Cell 3
    # không cần thêm gì
    eval_record = {
        **record,
        "accuracy_correct": (pred == (gt or "").upper()) and pred != "",
        "bertscore_f1"    : bs_score,
        "judge"           : judge_result,
    }
    append_jsonl(CHECKPOINT_EVAL, eval_record)

    if judge_result.get("parse_error"):
        n_parse_errors += 1

print(f"Đã ghi {len(todo_records)} records vào {CHECKPOINT_EVAL}")
print(f"Judge parse errors: {n_parse_errors} / {len(todo_records)}")

Đã ghi 500 records vào /kaggle/working/checkpoint_eval.jsonl
Judge parse errors: 0 / 500


In [12]:
# ── Cell 8: Aggregate summary ───────────────────────────────────────────
import json, shutil
from utils import load_jsonl
from evaluator import compute_aggregate_summary

all_eval_records = load_jsonl(CHECKPOINT_EVAL)

# Sort theo thứ tự gốc rồi strip _original_position
eval_sorted = sorted(all_eval_records, key=lambda r: r.get("_original_position", 0))

summary = compute_aggregate_summary(eval_sorted)

with open(OUTPUT_SUMMARY, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

with open(OUTPUT_EVAL, "w", encoding="utf-8") as f:
    for r in eval_sorted:
        clean = {k: v for k, v in r.items() if k != "_original_position"}
        f.write(json.dumps(clean, ensure_ascii=False) + "\n")

acc = summary["accuracy"]
bs  = summary["bertscore"]
jdg = summary["llm_judge"]

print("\n" + "="*52)
print("  FINAL EVALUATION SUMMARY")
print("="*52)
print(f"  Samples evaluated       : {summary['n_samples']}")
print()
print(f"  [1] Accuracy            : {acc['accuracy']:.4f}")
print(f"      Correct / Total     : {acc['n_correct']} / {acc['n_total']}")
print(f"      Skipped             : {acc['n_skipped']}")
print()
print(f"  [2] BertScore F1 (mean) : {bs['mean_f1']}")
print(f"      std                 : {bs['std_f1']}")
print()
print(f"  [3] Rationale Recall    : {jdg['mean_rationale_recall']}")
print(f"      std                 : {jdg['std_rationale_recall']}")
print("="*52)
print(f"\n  Saved: {OUTPUT_EVAL}")
print(f"  Saved: {OUTPUT_SUMMARY}")


  FINAL EVALUATION SUMMARY
  Samples evaluated       : 1000

  [1] Accuracy            : 0.5912
      Correct / Total     : 590 / 998
      Skipped             : 2

  [2] BertScore F1 (mean) : 0.7194
      std                 : 0.0338

  [3] Rationale Recall    : 0.7957
      std                 : 0.2948

  Saved: /kaggle/working/results_eval.jsonl
  Saved: /kaggle/working/summary_metrics.json


In [13]:
# ── Cell 9: Generate HTML Report ────────────────────────────────────────
from report_generator import generate_report
import json

OUTPUT_REPORT = "/kaggle/working/eval_report.html"

# Load lại eval records đã sorted và stripped
eval_records = []
with open(OUTPUT_EVAL, encoding="utf-8") as f:
    for line in f:
        eval_records.append(json.loads(line))

with open(OUTPUT_SUMMARY, encoding="utf-8") as f:
    summary = json.load(f)

generate_report(
    summary     = summary,
    records     = eval_records,
    output_path = OUTPUT_REPORT,
    model_id    = eval_records[0].get("model_id") if eval_records else "unknown",
)

Report saved → /kaggle/working/eval_report.html
